# Notebook 05 — Feature Engineering

**Goal:** Generate standard time-series feature datasets across multiple lag configurations (1-lag, 2-lag, 4-lag, and 12-lag) for baseline modeling and walk-forward validation.

### Pipeline Overview
- Extract seasonality & cyclical time features
- Construct lag and rolling mean features for dengue cases
- Construct weather lag features
- Export 1-lag, 2-lag, 4-lag, and 12-lag parquet datasets without lookahead leakage.


### 1. Imports


In [1]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


### 2. Configuration & Paths


In [2]:
INPUT_PATH = Path("../data/gold/dengue_modeling.parquet")

OUTPUT_PATH_1LAGS = Path("../data/processed/dengue_features_1lags.parquet")
OUTPUT_PATH_2LAGS = Path("../data/processed/dengue_features_2lags.parquet")
OUTPUT_PATH_4LAGS = Path("../data/processed/dengue_features_4lags.parquet")
OUTPUT_PATH_12LAGS = Path("../data/processed/dengue_features_12lags.parquet")
OUTPUT_PATH_DEFAULT = Path("../data/processed/dengue_features.parquet")

OUTPUT_PATH_DEFAULT.parent.mkdir(parents=True, exist_ok=True)


### 3. Load Data & Verify Temporal Structure


In [3]:
df = pd.read_parquet(INPUT_PATH)

df["week_start"] = pd.to_datetime(df["week_start"])
df = df.sort_values(["district", "week_start"]).reset_index(drop=True)

print(f"Raw dataset shape: {df.shape}")
print(f"Districts count: {df['district'].nunique()}")
print(f"Date span: {df['week_start'].min().date()} to {df['week_start'].max().date()}")
print(f"Missing target cases: {df['cases'].isna().sum()}")


Raw dataset shape: (26416, 17)
Districts count: 26
Date span: 2007-01-01 to 2026-06-29
Missing target cases: 26


### 4. Seasonality & Cyclical Features


In [4]:
# Extract temporal components
df["month"] = df["week_start"].dt.month
df["week_of_year"] = df["week_start"].dt.isocalendar().week.astype(int)

# Standard cyclical transformations (month & week)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["week_sin"] = np.sin(2 * np.pi * df["week_of_year"] / 53)
df["week_cos"] = np.cos(2 * np.pi * df["week_of_year"] / 53)

base_weather_cols = [
    "rainfall_mm", "temperature_mean", "temperature_max",
    "temperature_min", "humidity_mean", "wind_speed_mean", "dtr_mean"
]
base_temporal_cols = ["month", "week_of_year", "month_sin", "month_cos", "week_sin", "week_cos"]


### 5. Dataset 1: 1-Lag Pipeline (Short-Horizon)


In [5]:
df_1 = df.copy()

# Dengue target lag 1
df_1["cases_lag_1"] = df_1.groupby("district")["cases"].shift(1)

# Weather lag 1
for col in ["rainfall_mm", "temperature_mean", "humidity_mean"]:
    if col in df_1.columns:
        df_1[f"{col}_lag_1"] = df_1.groupby("district")[col].shift(1)

feature_columns_1 = [
    "cases_lag_1",
    *base_temporal_cols,
    *base_weather_cols,
    "rainfall_mm_lag_1",
    "temperature_mean_lag_1",
    "humidity_mean_lag_1",
]
feature_columns_1 = [c for c in feature_columns_1 if c in df_1.columns]

trainable_df_1 = df_1.dropna(subset=feature_columns_1).query("cases.notna()").copy()
print(f"1-Lag Trainable Shape: {trainable_df_1.shape} | Features: {len(feature_columns_1)}")


1-Lag Trainable Shape: (26338, 27) | Features: 17


### 6. Dataset 2: 2-Lag Pipeline (Streamlined Multi-Lag)


In [6]:
df_2 = df.copy()

# Dengue target lags 1, 2
for lag in [1, 2]:
    df_2[f"cases_lag_{lag}"] = df_2.groupby("district")["cases"].shift(lag)

# Rolling mean of cases (shift 1 to prevent leakage)
df_2["cases_roll_mean_2"] = df_2.groupby("district")["cases"].transform(
    lambda s: s.shift(1).rolling(2, min_periods=1).mean()
)

# Weather lags 1, 2
for col in ["rainfall_mm", "temperature_mean", "humidity_mean"]:
    if col in df_2.columns:
        for lag in [1, 2]:
            df_2[f"{col}_lag_{lag}"] = df_2.groupby("district")[col].shift(lag)

feature_columns_2 = [
    "cases_lag_1",
    "cases_lag_2",
    "cases_roll_mean_2",
    *base_temporal_cols,
    *base_weather_cols,
    "rainfall_mm_lag_1",
    "rainfall_mm_lag_2",
    "temperature_mean_lag_1",
    "temperature_mean_lag_2",
    "humidity_mean_lag_1",
    "humidity_mean_lag_2",
]
feature_columns_2 = [c for c in feature_columns_2 if c in df_2.columns]

trainable_df_2 = df_2.dropna(subset=feature_columns_2).query("cases.notna()").copy()
print(f"2-Lag Trainable Shape: {trainable_df_2.shape} | Features: {len(feature_columns_2)}")


2-Lag Trainable Shape: (26286, 32) | Features: 22


### 7. Dataset 3: 4-Lag Pipeline (Monthly Memory)


In [7]:
df_4 = df.copy()

# Dengue target lags 1, 2, 3, 4
for lag in [1, 2, 3, 4]:
    df_4[f"cases_lag_{lag}"] = df_4.groupby("district")["cases"].shift(lag)

# Rolling mean of cases 4
df_4["cases_roll_mean_4"] = df_4.groupby("district")["cases"].transform(
    lambda s: s.shift(1).rolling(4, min_periods=4).mean()
)

# Weather lags 1, 2, 4
for col in ["rainfall_mm", "temperature_mean", "humidity_mean"]:
    if col in df_4.columns:
        for lag in [1, 2, 4]:
            df_4[f"{col}_lag_{lag}"] = df_4.groupby("district")[col].shift(lag)

feature_columns_4 = [
    "cases_lag_1",
    "cases_lag_2",
    "cases_lag_3",
    "cases_lag_4",
    "cases_roll_mean_4",
    *base_temporal_cols,
    *base_weather_cols,
    "rainfall_mm_lag_1",
    "rainfall_mm_lag_2",
    "rainfall_mm_lag_4",
    "temperature_mean_lag_1",
    "temperature_mean_lag_2",
    "temperature_mean_lag_4",
    "humidity_mean_lag_1",
    "humidity_mean_lag_2",
    "humidity_mean_lag_4",
]
feature_columns_4 = [c for c in feature_columns_4 if c in df_4.columns]

trainable_df_4 = df_4.dropna(subset=feature_columns_4).query("cases.notna()").copy()
print(f"4-Lag Trainable Shape: {trainable_df_4.shape} | Features: {len(feature_columns_4)}")


4-Lag Trainable Shape: (26182, 37) | Features: 27


### 8. Dataset 4: 12-Lag Pipeline (Quarterly Long Memory)


In [8]:
df_12 = df.copy()

# Dengue target lags 1, 2, 3, 4, 8, 12
for lag in [1, 2, 3, 4, 8, 12]:
    df_12[f"cases_lag_{lag}"] = df_12.groupby("district")["cases"].shift(lag)

# Rolling averages 4, 8, 12
for window in [4, 8, 12]:
    df_12[f"cases_roll_mean_{window}"] = df_12.groupby("district")["cases"].transform(
        lambda s: s.shift(1).rolling(window, min_periods=window).mean()
    )

# Weather lags 1, 2, 4
for col in ["rainfall_mm", "temperature_mean", "humidity_mean"]:
    if col in df_12.columns:
        for lag in [1, 2, 4]:
            df_12[f"{col}_lag_{lag}"] = df_12.groupby("district")[col].shift(lag)

feature_columns_12 = [
    "cases_lag_1",
    "cases_lag_2",
    "cases_lag_3",
    "cases_lag_4",
    "cases_lag_8",
    "cases_lag_12",
    "cases_roll_mean_4",
    "cases_roll_mean_8",
    "cases_roll_mean_12",
    *base_temporal_cols,
    *base_weather_cols,
    "rainfall_mm_lag_1",
    "rainfall_mm_lag_2",
    "rainfall_mm_lag_4",
    "temperature_mean_lag_1",
    "temperature_mean_lag_2",
    "temperature_mean_lag_4",
    "humidity_mean_lag_1",
    "humidity_mean_lag_2",
    "humidity_mean_lag_4",
]
feature_columns_12 = [c for c in feature_columns_12 if c in df_12.columns]

trainable_df_12 = df_12.dropna(subset=feature_columns_12).query("cases.notna()").copy()
print(f"12-Lag Trainable Shape: {trainable_df_12.shape} | Features: {len(feature_columns_12)}")


12-Lag Trainable Shape: (25766, 41) | Features: 31


### 9. Comparison & Dataset Summary


In [9]:
summary_df = pd.DataFrame([
    {"Dataset": "1-Lag Dataset", "Panel Rows": len(df), "Trainable Rows": len(trainable_df_1), "Excluded Rows": len(df) - len(trainable_df_1), "Feature Count": len(feature_columns_1)},
    {"Dataset": "2-Lag Dataset", "Panel Rows": len(df), "Trainable Rows": len(trainable_df_2), "Excluded Rows": len(df) - len(trainable_df_2), "Feature Count": len(feature_columns_2)},
    {"Dataset": "4-Lag Dataset", "Panel Rows": len(df), "Trainable Rows": len(trainable_df_4), "Excluded Rows": len(df) - len(trainable_df_4), "Feature Count": len(feature_columns_4)},
    {"Dataset": "12-Lag Dataset", "Panel Rows": len(df), "Trainable Rows": len(trainable_df_12), "Excluded Rows": len(df) - len(trainable_df_12), "Feature Count": len(feature_columns_12)},
])
print(summary_df.to_string(index=False))


       Dataset  Panel Rows  Trainable Rows  Excluded Rows  Feature Count
 1-Lag Dataset       26416           26338             78             17
 2-Lag Dataset       26416           26286            130             22
 4-Lag Dataset       26416           26182            234             27
12-Lag Dataset       26416           25766            650             31


### 10. Save Processed Feature Datasets


In [10]:
meta_cols = ["district", "week_start", "cases"]

# 1. Save 1-lag
trainable_df_1[meta_cols + feature_columns_1].to_parquet(OUTPUT_PATH_1LAGS, index=False)
print(f"Saved: {OUTPUT_PATH_1LAGS}")

# 2. Save 2-lag
trainable_df_2[meta_cols + feature_columns_2].to_parquet(OUTPUT_PATH_2LAGS, index=False)
print(f"Saved: {OUTPUT_PATH_2LAGS}")

# 3. Save 4-lag
trainable_df_4[meta_cols + feature_columns_4].to_parquet(OUTPUT_PATH_4LAGS, index=False)
print(f"Saved: {OUTPUT_PATH_4LAGS}")

# 4. Save 12-lag
trainable_df_12[meta_cols + feature_columns_12].to_parquet(OUTPUT_PATH_12LAGS, index=False)
print(f"Saved: {OUTPUT_PATH_12LAGS}")

# 5. Save Default (2-lag)
trainable_df_2[meta_cols + feature_columns_2].to_parquet(OUTPUT_PATH_DEFAULT, index=False)
print(f"Saved Default: {OUTPUT_PATH_DEFAULT}")


Saved: ../data/processed/dengue_features_1lags.parquet
Saved: ../data/processed/dengue_features_2lags.parquet
Saved: ../data/processed/dengue_features_4lags.parquet
Saved: ../data/processed/dengue_features_12lags.parquet
Saved Default: ../data/processed/dengue_features.parquet
